In [31]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
import pandas as pd
from tqdm import tqdm

# Model config
# MODEL_NAME = "google/mt5-base"
# MODEL_NAME = "google-t5/t5-base"
MODEL_NAME = "google-t5/t5-small"
TASK_PREFIX = "korrigiere grammatik: "

# Simple Hyperparameters
LEARNING_RATE = 1e-4
EPOCHS = 5
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

In [32]:
# Mock Data
# MOCK_DATA = [
#     ("Das ist ein fehler.", "Das ist ein Fehler."),
#     ("er geht schule.", "Er geht zur Schule."),
#     ("Sie lauft schnell.", "Sie läuft schnell."),
#     ("Mein vater ist artz.", "Mein Vater ist Arzt."),
#     ("ich habe ein buch gelesen", "Ich habe ein Buch gelesen."),
#     ("Wir sind auf dem weg.", "Wir sind auf dem Weg."),
#     ("die katze schläft.", "Die Katze schläfT."), # Note: A small error to fix
#     ("er hat kein geld.", "Er hat kein Geld."),
# ]

# df = pd.read_csv("data/chapter1_corrupted.csv", sep=";")
df = pd.read_csv("data/news_data_corrupted_small.csv", sep=";")

# Create list of tuples from dataframe (corrupted, correct)
data_tuples = list(zip(df['de_corrupted'].tolist(), df['de_correct'].tolist()))

TRAIN_DATA = data_tuples[:int(len(data_tuples) * 0.8)]
VAL_DATA = data_tuples[int(len(data_tuples) * 0.8):]

In [33]:
class GermanGECDataset(Dataset):
    """
    A custom PyTorch Dataset for German Grammatical Error Correction.
    It tokenizes data on the fly.
    """
    def __init__(self, data: List[tuple], tokenizer: AutoTokenizer, prefix: str, max_input_len: int, max_target_len: int):
        self.data = data
        self.tokenizer = tokenizer
        self.prefix = prefix
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        incorrect_text, correct_text = self.data[index]
        
        # Prepare the input text (with prefix)
        input_text = self.prefix + incorrect_text

        # Tokenize the input
        # We don't pad here; the collator will handle it.
        tokenized_input = self.tokenizer(
            input_text,
            max_length=self.max_input_len,
            truncation=True,
            return_tensors="pt"
        )
        
        # Tokenize the target
        # We don't pad here; the collator will handle it.
        tokenized_target = self.tokenizer(
            text_target=correct_text,
            max_length=self.max_target_len,
            truncation=True,
            return_tensors="pt"
        )

        # Squeeze to remove the batch dimension (which is 1)
        # The DataLoader will add it back.
        input_ids = tokenized_input["input_ids"].squeeze(0)
        attention_mask = tokenized_input["attention_mask"].squeeze(0)
        labels = tokenized_target["input_ids"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }


In [34]:
class GECDataCollator:
    """
    A custom data collator. This is crucial for a manual loop.
    It pads batches dynamically to the longest sequence in that batch.
    """
    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        
        # --- 1. Pad Input IDs and Attention Masks ---
        
        # Get all input_ids from the batch
        input_ids_list = [item["input_ids"] for item in batch]
        
        # Pad them to the longest sequence in this batch
        # The tokenizer's pad method is excellent for this.
        # It handles padding, truncation, and returns a tensor.
        padded_inputs = self.tokenizer.pad(
            {"input_ids": input_ids_list},
            padding="longest",
            return_tensors="pt"
        )
        
        # --- 2. Pad Labels ---
        
        # Get all labels from the batch
        labels_list = [item["labels"] for item in batch]
        
        # Pad them to the longest sequence in this batch
        padded_labels = self.tokenizer.pad(
            {"input_ids": labels_list},
            padding="longest",
            return_tensors="pt"
        ).get("input_ids") # We only want the input_ids from this
        
        # --- 3. Mask Padded Labels ---
        
        # This is CRITICAL. The model should NOT calculate loss
        # on padding tokens in the labels.
        # We replace the pad_token_id (e.g., 0) with -100.
        # PyTorch's CrossEntropyLoss automatically ignores -100.
        padded_labels[padded_labels == self.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": padded_inputs["input_ids"],
            "attention_mask": padded_inputs["attention_mask"],
            "labels": padded_labels
        }


In [35]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """One full training pass over the dataset."""
    model.train()  # Set model to training mode
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Clear previous gradients
        optimizer.zero_grad()
        
        # Forward pass
        # When 'labels' are provided, T5 automatically computes 
        # the cross-entropy loss.
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        # Get the loss
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Update learning rate scheduler
        scheduler.step()
        
        total_loss += loss.item()

        # Update progress bar with current batch loss
        progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
        
    return total_loss / len(dataloader)


In [36]:
def evaluate_epoch(model, dataloader, device):
    """One full evaluation pass."""
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    
    # We don't need to track gradients during evaluation
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Evaluating")
        for batch in progress_bar:
            # Move batch to device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            # Get the loss
            loss = outputs.loss
            total_loss += loss.item()

            # Update progress bar with current batch loss
            progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
            
    return total_loss / len(dataloader)


In [37]:
def generate_correction(model, tokenizer, prefix, sentence, device):
    """Generate a correction for a single sentence."""
    model.eval() # Set model to eval mode
    
    input_text = prefix + sentence
    
    # Tokenize
    tokenized_input = tokenizer(input_text, return_tensors="pt")
    input_ids = tokenized_input["input_ids"].to(device)
    attention_mask = tokenized_input["attention_mask"].to(device)
    
    # Generate output
    # You can experiment with beam search, top-k, etc.
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4, # Use beam search for better quality
            early_stopping=True
        )
        
    # Decode the generated IDs back to text
    corrected_text = tokenizer.decode(
        generated_ids[0], 
        skip_special_tokens=True
    )
    
    return corrected_text


In [38]:
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

gec_collator = GECDataCollator(tokenizer=tokenizer)

train_dataset = GermanGECDataset(
    TRAIN_DATA, tokenizer, TASK_PREFIX, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH
)
val_dataset = GermanGECDataset(
    VAL_DATA, tokenizer, TASK_PREFIX, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=gec_collator,
    shuffle=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=gec_collator,
    shuffle=False
)

Loading tokenizer: google-t5/t5-small
Loading model: google-t5/t5-small
Loading model: google-t5/t5-small


In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Setup Learning Rate Scheduler
# Total number of training steps
num_training_steps = EPOCHS * len(train_dataloader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0, # You can set a warmup
    num_training_steps=num_training_steps
)

# The Training Loop
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    
    # Training
    train_loss = train_epoch(
        model, train_dataloader, optimizer, scheduler, device
    )
    print(f"  Training Loss:   {train_loss:.4f}")
    
    # Evaluation
    val_loss = evaluate_epoch(
        model, val_dataloader, device
    )
    print(f"  Validation Loss: {val_loss:.4f}")
    
    # Checkpoint: Test generation
    print("--- Checking Generation ---")
    test_sentence = "er geht schule."
    correction = generate_correction(
        model, tokenizer, TASK_PREFIX, test_sentence, device
    )
    print(f"  Input:    '{test_sentence}'")
    print(f"  Output:   '{correction}'")
    print("---------------------------")

print("\n--- Training Finished ---")


Using device: cuda

Epoch 1/5


Training:   0%|          | 0/2000 [00:00<?, ?it/s]You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Training: 100%|██████████| 2000/2000 [02:14<00:00, 14.84it/s, batch_loss=1.0785]


  Training Loss:   1.1640


Evaluating: 100%|██████████| 500/500 [00:10<00:00, 45.93it/s, batch_loss=1.4105]



  Validation Loss: 0.9367
--- Checking Generation ---
  Input:    'er geht schule.'
  Output:   'Es geht um die Schule.'
---------------------------

Epoch 2/5


...

  Validation Loss: 0.8466
--- Checking Generation ---
  Input:    'er geht schule.'
  Output:   'Es geht um eine schulische Ausbildung.'
---------------------------

Epoch 5/5


Training: 100%|██████████| 2000/2000 [02:15<00:00, 14.77it/s, batch_loss=1.0147]


  Training Loss:   0.7914


Evaluating: 100%|██████████| 500/500 [00:10<00:00, 45.60it/s, batch_loss=1.4198]



  Validation Loss: 0.8469
--- Checking Generation ---
  Input:    'er geht schule.'
  Output:   'Es geht um eine schule.'
---------------------------

--- Training Finished ---


In [40]:
# Save the fine-tuned model
model.save_pretrained("./my_finetuned_t5_german_gec")
tokenizer.save_pretrained("./my_finetuned_t5_german_gec")
print("Model saved to ./my_finetuned_t5_german_gec")

Model saved to ./my_finetuned_t5_german_gec
